In [ ]:
# Deep Learning Image Classification using TensorFlow
# CIFAR-10 Dataset with Transfer Learning

# =========================
# 1. Import Libraries
# =========================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# =========================
# 2. Load CIFAR-10 Dataset
# =========================

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Class names
class_names = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
]

# =========================
# 3. Preprocessing
# =========================

# Resize images for MobileNetV2
IMG_SIZE = 96

x_train_resized = tf.image.resize(x_train, (IMG_SIZE, IMG_SIZE))
x_test_resized = tf.image.resize(x_test, (IMG_SIZE, IMG_SIZE))

# Normalize images
x_train_resized = x_train_resized / 255.0
x_test_resized = x_test_resized / 255.0

# =========================
# 4. Data Augmentation
# =========================

datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

datagen.fit(x_train_resized)

# =========================
# 5. Load Pretrained Model
# =========================

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze pretrained layers
base_model.trainable = False

# =========================
# 6. Build Model
# =========================

model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.3),

    layers.Dense(10, activation='softmax')

])

# =========================
# 7. Compile Model
# =========================

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# =========================
# 8. Train Model
# =========================

history = model.fit(

    datagen.flow(x_train_resized, y_train, batch_size=32),

    epochs=5,

    validation_data=(x_test_resized, y_test)

)

# =========================
# 9. Plot Training Curves
# =========================

plt.figure(figsize=(10,5))

# Accuracy Plot
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend()
plt.show()

# =========================
# 10. Evaluate Model
# =========================

y_pred_prob = model.predict(x_test_resized)

y_pred = np.argmax(y_pred_prob, axis=1)

y_true = y_test.flatten()

# Metrics
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average='weighted'
)

recall = recall_score(
    y_true,
    y_pred,
    average='weighted'
)

f1 = f1_score(
    y_true,
    y_pred,
    average='weighted'
)

print("\nModel Evaluation")
print("-" * 30)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# Classification Report
print("\nClassification Report")
print(classification_report(y_true, y_pred))

# =========================
# 11. Confusion Matrix
# =========================

cm = confusion_matrix(y_true, y_pred)

print("\nConfusion Matrix")
print(cm)

# =========================
# 12. Save Model
# =========================

model.save("image_classifier_model.h5")

print("\nModel saved successfully!")

# =========================
# 13. Run Inference
# =========================

# Example prediction on one image

sample_image = x_test_resized[0]

sample_image = np.expand_dims(sample_image, axis=0)

prediction = model.predict(sample_image)

predicted_class = np.argmax(prediction)

print("\nPredicted Class:",
      class_names[predicted_class])

print("Actual Class:",
      class_names[y_true[0]])